# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata.to_json()
print(f"Dataset title: {metadata['name']}")
print(f"Description: {metadata['description']}")
print(f"License: {metadata['license']}")
print(f"Published: {metadata['datePublished']}")


## 2. Data Overview
Review available record sets, fields, and their IDs. Record sets are referenced by their unique `@id`.

In [ ]:
# Access the recordsets present in the dataset

# Record sets are listed under the `record_sets` property:
record_sets = dataset.record_sets
print(f"Number of record sets: {len(record_sets)}\n")
print("Available Record Sets (by @id and name):")
for rs in record_sets:
    print(f"  @id: {rs.id}, name: {rs.name}")

# Display fields for each record set
print("\nField @id and name for each record set:")
for rs in record_sets:
    print(f"\nRecord Set: {rs.id} ({rs.name})")
    for field in rs.fields:
        print(f"  Field @id: {field.id}, name: {field.name}, dataType: {field.data_type}")
    if len(rs.fields) == 0:
        print("  No fields defined.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# List the record set @id's for convenience
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

# Load all available record sets into dataframes
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df

if record_set_ids:
    main_record_set_id = record_set_ids[0]
    print(f"Columns in the DataFrame for record set '{main_record_set_id}':")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No record sets found.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# For demonstration, select numeric fields to filter and normalize.
# We'll automatically search for common numeric field names (e.g., 'MW', 'Coverage', 'PeptideCount', etc.)

import numpy as np

df = dataframes.get(main_record_set_id)
if df is not None:
    # Try to find a numeric field
    possible_numeric_fields = [
        col for col in df.columns if str(df[col].dtype) in ('float64', 'int64', 'float32', 'int32')
    ]
    if len(possible_numeric_fields) == 0:
        # Try to guess from column names (case-insensitive)
        substrings = ['mw', 'weight', 'coverage', 'abundance', 'count', 'score']
        for col in df.columns:
            for sub in substrings:
                if sub in col.lower():
                    possible_numeric_fields.append(col)
    if len(possible_numeric_fields) > 0:
        numeric_field = possible_numeric_fields[0]
        # Try to coerce to numeric
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
        # Example threshold
        threshold = df[numeric_field].mean() if pd.notnull(df[numeric_field].mean()) else 0
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())
        # Normalize the numeric field
        filtered_df[f"{numeric_field}_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        )
        print(f"\nNormalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        # Try grouping by another field, e.g. description, if present
        group_field = None
        for col in df.columns:
            if 'description' in col.lower() or 'accession' in col.lower() or 'id' == col.lower():
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nGrouped data by {group_field} (mean of {numeric_field}):")
            display(grouped_df.head())
    else:
        print("No numeric field found in the main record set.")
else:
    print("No main dataframe found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution of the numeric field
if df is not None and 'numeric_field' in locals():
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # Boxplot for the field (if enough data)
    plt.figure(figsize=(5, 5))
    sns.boxplot(y=df[numeric_field].dropna())
    plt.title(f"Boxplot of {numeric_field}")
    plt.ylabel(numeric_field)
    plt.show()

    # If grouping and more than one group, show means
    if 'group_field' in locals() and group_field is not None:
        grouped = df.groupby(group_field)[numeric_field].mean().reset_index()
        plt.figure(figsize=(10, 4))
        sns.barplot(x=group_field, y=numeric_field, data=grouped)
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xticks(rotation=90)
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load and explore the FAIR<sup>2</sup> mass spectrometry dataset using `mlcroissant`. We reviewed the available record sets by their `@id`, explored the data fields, loaded the main record set into a DataFrame, and performed basic analysis and visualizations on numeric attributes.

Further exploration could include advanced statistical analysis, feature engineering, or applying machine learning analyses as appropriate to the biological context.